# 03 - Model Training

Цель: показать конфиг обучения, split, архитектуру и историю обучения.


In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))

DATA_SYNTHETIC = ROOT / "data" / "synthetic"
DATA_PROCESSED = ROOT / "data" / "processed"
REPORTS = ROOT / "reports"
FIGURES = REPORTS / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

print("Project root:", ROOT)


Project root: /Users/andrey/Documents/projects/COMPASS-AI


In [2]:
split_meta_path = DATA_PROCESSED / "split_metadata.json"
training_config_path = REPORTS / "training_config.json"
history_path = REPORTS / "training_history.csv"
checkpoint_path = ROOT / "models" / "compass_matching_model.pt"

with open(split_meta_path, encoding="utf-8") as file:
    split_meta = json.load(file)

with open(training_config_path, encoding="utf-8") as file:
    training_config = json.load(file)

print("split metadata:")
display(split_meta)

print("training config:")
display(training_config)

print("checkpoint exists:", checkpoint_path.exists(), checkpoint_path)


split metadata:


{'random_state': 42,
 'split_strategy': 'group_split_by_task_id',
 'train_size_target': 0.7,
 'val_size_target': 0.15,
 'test_size_target': 0.15,
 'splits': [{'split': 'train',
   'rows': 42052,
   'tasks': 1750,
   'employees': 36,
   'success_rate': 0.450775},
  {'split': 'val',
   'rows': 9024,
   'tasks': 375,
   'employees': 36,
   'success_rate': 0.44293},
  {'split': 'test',
   'rows': 8924,
   'tasks': 375,
   'employees': 36,
   'success_rate': 0.457642}],
 'leakage_check': {'task_id_overlap_train_val': 0,
  'task_id_overlap_train_test': 0,
  'task_id_overlap_val_test': 0}}

training config:


{'device': 'mps',
 'epochs': 40,
 'batch_size': 512,
 'learning_rate': 0.0007,
 'weight_decay': 0.0001,
 'hidden_dim': 256,
 'embedding_dim': 128,
 'dropout': 0.1,
 'patience': 8,
 'train_rows': 42052,
 'val_rows': 9024,
 'task_dim': 404,
 'employee_dim': 60,
 'pair_dim': 13}

checkpoint exists: True /Users/andrey/Documents/projects/COMPASS-AI/models/compass_matching_model.pt


In [3]:
train_df = pd.read_parquet(DATA_PROCESSED / "train.parquet")
val_df = pd.read_parquet(DATA_PROCESSED / "val.parquet")
test_df = pd.read_parquet(DATA_PROCESSED / "test.parquet")

split_summary = pd.DataFrame(
    [
        {
            "split": "train",
            "rows": len(train_df),
            "tasks": train_df["task_id"].nunique(),
            "success_rate": train_df["success_label"].mean(),
        },
        {
            "split": "val",
            "rows": len(val_df),
            "tasks": val_df["task_id"].nunique(),
            "success_rate": val_df["success_label"].mean(),
        },
        {
            "split": "test",
            "rows": len(test_df),
            "tasks": test_df["task_id"].nunique(),
            "success_rate": test_df["success_label"].mean(),
        },
    ]
)

display(split_summary)


,split,rows,tasks,success_rate
0,train,42052,1750,0.450775
1,val,9024,375,0.442930
2,test,8924,375,0.457642


In [4]:
import inspect

import torch

from src.models.matching_net import MatchingNetConfig, TaskEmployeeMatchingNet

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False,
)

config_source = checkpoint.get("model_config", {})
if not config_source:
    config_source = training_config.get("model_config", {})
if not config_source:
    config_source = training_config

aliases = {
    "task_input_dim": training_config.get("task_dim"),
    "employee_input_dim": training_config.get("employee_dim"),
    "pair_input_dim": training_config.get("pair_dim"),
    "task_dim": training_config.get("task_dim"),
    "employee_dim": training_config.get("employee_dim"),
    "pair_dim": training_config.get("pair_dim"),
    "hidden_dim": training_config.get("hidden_dim", 256),
    "embedding_dim": training_config.get("embedding_dim", 128),
    "dropout": training_config.get("dropout", 0.1),
}
aliases.update(config_source)

config_kwargs = {}
for name in inspect.signature(MatchingNetConfig).parameters:
    if name in aliases and aliases[name] is not None:
        config_kwargs[name] = aliases[name]

print("MatchingNetConfig kwargs:")
display(config_kwargs)

model_config = MatchingNetConfig(**config_kwargs)
model = TaskEmployeeMatchingNet(model_config)

model


MatchingNetConfig kwargs:


{'task_input_dim': 404,
 'employee_input_dim': 60,
 'pair_input_dim': 13,
 'task_embedding_dim': 128,
 'employee_embedding_dim': 128,
 'hidden_dim': 256,
 'dropout': 0.1}

TaskEmployeeMatchingNet(
  (task_encoder): TaskEncoder(
    (network): Sequential(
      (0): Linear(in_features=404, out_features=256, bias=True)
      (1): LayerNorm((256,), eps=1e-05, elementwise_affine=True, bias=True)
      (2): ReLU()
      (3): Dropout(p=0.1, inplace=False)
      (4): Linear(in_features=256, out_features=128, bias=True)
      (5): LayerNorm((128,), eps=1e-05, elementwise_affine=True, bias=True)
      (6): ReLU()
    )
  )
  (employee_encoder): EmployeeEncoder(
    (network): Sequential(
      (0): Linear(in_features=60, out_features=128, bias=True)
      (1): LayerNorm((128,), eps=1e-05, elementwise_affine=True, bias=True)
      (2): ReLU()
      (3): Dropout(p=0.1, inplace=False)
      (4): Linear(in_features=128, out_features=128, bias=True)
      (5): LayerNorm((128,), eps=1e-05, elementwise_affine=True, bias=True)
      (6): ReLU()
    )
  )
  (matching_block): MatchingBlock(
    (network): Sequential(
      (0): Linear(in_features=525, out_features=256, bia

In [5]:
history = pd.read_csv(history_path)
display(history.head())


,epoch,train_loss,val_loss,val_roc_auc
0,1,0.551481,0.471854,0.857706
1,2,0.479879,0.469478,0.857085
2,3,0.474325,0.456176,0.865802
3,4,0.468639,0.457526,0.865519
4,5,0.467210,0.453581,0.867048


In [6]:
import plotly.express as px

fig = px.line(
    history,
    x="epoch",
    y=["train_loss", "val_loss"],
)
fig.update_layout(title="Training and validation loss")
fig.show()
fig.write_html(FIGURES / "training_loss.html")


In [7]:
import plotly.express as px

if "val_roc_auc" in history.columns:
    fig = px.line(history, x="epoch", y="val_roc_auc")
    fig.update_layout(title="Validation ROC-AUC")
    fig.show()
    fig.write_html(FIGURES / "training_val_roc_auc.html")
